In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l1.2 Matrix multiplication

> 🎯 **Goal:** Understand the matrix-multiply shape rule so you can predict the output shape of any layer in your head.

**Why this matters.** Matrix multiplication (matmul for short) is the workhorse of every neural network. When a transformer "mixes" information, projects an embedding, or computes attention, it is almost always doing a matmul under the hood. If you can track shapes through a matmul, you can debug 90 percent of the errors you will ever hit, because most of them are shape mismatches.

**The intuition.** Picture a matmul as combining two tables. The left table has some rows; the right table has some columns. To fill in one cell of the answer, you slide a row of the left table across a column of the right table, multiply the lined-up pairs, and add them up. For that sliding to work, the row and the column must be the same length. That shared length is the dimension that "cancels".

**The mechanics.** The rule, written with the `@` operator that PyTorch uses for matmul:

`(m, k) @ (k, n)` produces `(m, n)`.

Read it left to right. The left tensor is `m` rows by `k` columns. The right tensor is `k` rows by `n` columns. The two inner numbers (both `k`) must be equal; if they are not, PyTorch refuses with a shape error. Those inner `k`s cancel, and the result keeps the two outer numbers: `m` rows by `n` columns. A handy mnemonic: the outer dimensions survive, the inner dimensions must agree and then vanish.

In the code, `torch.randn(2, 3)` makes a 2-by-3 tensor of random floats (`randn` means "random, normally distributed"). So we expect `(2, 3) @ (3, 4)` to give `(2, 4)`.

In [ ]:
a = torch.randn(2, 3)
b = torch.randn(3, 4)
c = a @ b
print("(2,3) @ (3,4) ->", tuple(c.shape))

**What just happened.** The inner `3`s matched and cancelled, leaving the outer `2` and `4`, so `c` has shape `(2, 4)`, exactly as the rule predicted. Get in the habit of tracing shapes through every matmul on paper before you run anything; most bugs disappear before they even execute.

⚠️ **Common pitfalls**
- Mismatched inner dimensions. `(2, 3) @ (4, 5)` fails because the inner numbers `3` and `4` are not equal. The fix is usually to transpose one tensor (swap its rows and columns) so the inner dimensions line up.
- Forgetting that order matters. `a @ b` is not the same as `b @ a`. Unlike multiplying plain numbers, swapping the operands changes the shape (and usually fails outright).

🏋️ **Try it yourself.** Change `b` to `torch.randn(4, 4)` and rerun. You will get a shape error because `(2, 3) @ (4, 4)` has mismatched inner dimensions (`3` versus `4`). Read the error message closely, then fix it by changing `b` back to start with `3`, for example `torch.randn(3, 5)`, and predict the output shape before running.

In [ ]:
assert tuple(c.shape) == (2, 4)